# Notebook 02 — Loss Functions: L1, L2, and Huber Loss Formulations

**Session:** Linear Regression — L1, L2, and Huber Losses (Level 1)  
**Scheduled Date:** 2026-11-23 · **Module:** 02 — Calculus and Linear Algebra

---

## 1. Overview and Objectives

In linear regression, the choice of loss function $\ell(\hat{y}, y)$ dictates how the model balances
average accuracy against sensitivity to extreme anomalies:
1. **L2 Loss (Mean Squared Error):** Quadratic penalization, continuously differentiable, but sensitive to outliers.
2. **L1 Loss (Mean Absolute Error):** Linear penalization, robust to outliers, non-differentiable at residual $r=0$.
3. **Huber Loss (Smooth L1):** Quadratic for small errors ($|r| \le \delta$), linear for large errors ($|r| > \delta$).

We will:
- Implement all three losses and their analytical sub-gradients.
- Perform finite-difference numerical gradient checks to verify analytical formulas.
- Evaluate the impact of outliers on model convergence.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["figure.figsize"] = (8, 4.5)
plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

## 2. Mathematical Definitions

Given residual $r = \hat{y} - y = (wx + b) - y$:

### Mean Squared Error (L2 Loss)
$$\ell_{\text{MSE}}(r) = r^2$$
$$\frac{\partial \ell}{\partial w} = 2 x r, \qquad \frac{\partial \ell}{\partial b} = 2 r$$

### Mean Absolute Error (L1 Loss)
$$\ell_{\text{MAE}}(r) = |r|$$
$$\frac{\partial \ell}{\partial w} = x \cdot \text{sign}(r), \qquad \frac{\partial \ell}{\partial b} = \text{sign}(r)$$
where $\text{sign}(r) = +1$ if $r > 0$, $-1$ if $r < 0$, and $0$ (subgradient convention) if $r = 0$.

### Huber Loss (Smooth L1 Loss)
$$\ell_{\text{Huber}}(r; \delta) = \begin{cases} \frac{1}{2} r^2 & \text{if } |r| \le \delta \\ \delta (|r| - \frac{1}{2}\delta) & \text{if } |r| > \delta \end{cases}$$
$$\frac{\partial \ell_{\text{Huber}}}{\partial r} = \begin{cases} r & \text{if } |r| \le \delta \\ \delta \cdot \text{sign}(r) & \text{if } |r| > \delta \end{cases}$$
$$\frac{\partial \ell}{\partial w} = x \cdot \frac{\partial \ell}{\partial r}, \qquad \frac{\partial \ell}{\partial b} = \frac{\partial \ell}{\partial r}$$


In [ ]:
def loss_mse(r: np.ndarray | float) -> np.ndarray | float:
    return r**2


def grad_mse(x: float, r: float) -> tuple[float, float]:
    return 2.0 * x * r, 2.0 * r


def loss_mae(r: np.ndarray | float) -> np.ndarray | float:
    return np.abs(r)


def grad_mae(x: float, r: float) -> tuple[float, float]:
    sign = float(np.sign(r))
    return x * sign, sign


def loss_huber(r: np.ndarray | float, delta: float = 1.0) -> np.ndarray | float:
    abs_r = np.abs(r)
    is_quadratic = abs_r <= delta
    quadratic = 0.5 * (r**2)
    linear = delta * (abs_r - 0.5 * delta)
    return np.where(is_quadratic, quadratic, linear)


def grad_huber(x: float, r: float, delta: float = 1.0) -> tuple[float, float]:
    dr = r if abs(r) <= delta else float(delta * np.sign(r))
    return x * dr, dr

## 3. Finite-Difference Numerical Gradient Checking

To verify our analytical derivatives, we compute the centered two-point finite difference:
$$\frac{\partial \ell}{\partial \theta} \approx \frac{\ell(\theta + \epsilon) - \ell(\theta - \epsilon)}{2\epsilon}$$
and measure the relative error:
$$\text{rel\_error} = \frac{|\nabla_{\text{analytic}} - \nabla_{\text{numerical}}|}{|\nabla_{\text{analytic}}| + |\nabla_{\text{numerical}}| + 10^{-12}}$$


In [ ]:
def check_gradient(
    loss_fn, grad_fn, w: float, b: float, x: float, y: float, eps: float = 1e-6
) -> dict[str, float]:
    # Analytical gradient
    r = (w * x + b) - y
    dw_ana, db_ana = grad_fn(x, r)

    # Numerical w gradient
    r_w_plus = ((w + eps) * x + b) - y
    r_w_minus = ((w - eps) * x + b) - y
    dw_num = (loss_fn(r_w_plus) - loss_fn(r_w_minus)) / (2 * eps)

    # Numerical b gradient
    r_b_plus = (w * x + (b + eps)) - y
    r_b_minus = (w * x + (b - eps)) - y
    db_num = (loss_fn(r_b_plus) - loss_fn(r_b_minus)) / (2 * eps)

    err_w = abs(dw_ana - dw_num) / (abs(dw_ana) + abs(dw_num) + 1e-12)
    err_b = abs(db_ana - db_num) / (abs(db_ana) + abs(db_num) + 1e-12)

    return {"err_w": float(err_w), "err_b": float(err_b)}


# Verify on sample x=6.7, y=9.1, w=1.2, b=0.3
x_test, y_test, w_test, b_test = 6.7, 9.1, 1.2, 0.3

res_mse = check_gradient(loss_mse, grad_mse, w_test, b_test, x_test, y_test)
res_mae = check_gradient(loss_mae, grad_mae, w_test, b_test, x_test, y_test)
res_huber = check_gradient(
    lambda r: loss_huber(r, delta=1.5),
    lambda x, r: grad_huber(x, r, delta=1.5),
    w_test,
    b_test,
    x_test,
    y_test,
)

print(f"MSE Gradient Rel Errors:   w={res_mse['err_w']:.2e}, b={res_mse['err_b']:.2e}")
print(f"MAE Gradient Rel Errors:   w={res_mae['err_w']:.2e}, b={res_mae['err_b']:.2e}")
print(f"Huber Gradient Rel Errors: w={res_huber['err_w']:.2e}, b={res_huber['err_b']:.2e}")

assert res_mse["err_w"] < 1e-5 and res_mse["err_b"] < 1e-5
assert res_mae["err_w"] < 1e-5 and res_mae["err_b"] < 1e-5
assert res_huber["err_w"] < 1e-5 and res_huber["err_b"] < 1e-5
print("\nAll analytical gradient checks passed!")

## 4. Visualizing Loss Curves & Gradient Profiles

Notice how the derivative $\frac{\partial L}{\partial r}$ saturates for MAE and Huber loss,
preventing outliers from exerting unbounded gradient magnitudes.


In [ ]:
r_vals = np.linspace(-4, 4, 400)
delta_param = 1.2

l_mse = loss_mse(r_vals)
l_mae = loss_mae(r_vals)
l_hub = loss_huber(r_vals, delta=delta_param)

# Derivative curves
g_mse = 2.0 * r_vals
g_mae = np.sign(r_vals)
g_hub = np.where(np.abs(r_vals) <= delta_param, r_vals, delta_param * np.sign(r_vals))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

ax1.plot(r_vals, l_mse, label="MSE (L2)", color="tab:blue", lw=2)
ax1.plot(r_vals, l_mae, label="MAE (L1)", color="tab:green", lw=2)
ax1.plot(r_vals, l_hub, label=f"Huber (δ={delta_param})", color="tab:red", lw=2, linestyle="--")
ax1.set_title("Loss Value vs. Residual r")
ax1.set_xlabel("Residual r = ŷ - y")
ax1.set_ylabel("Loss")
ax1.set_ylim(-0.2, 10)
ax1.legend()

ax2.plot(r_vals, g_mse, label="MSE Derivative", color="tab:blue", lw=2)
ax2.plot(r_vals, g_mae, label="MAE Derivative", color="tab:green", lw=2)
ax2.plot(
    r_vals,
    g_hub,
    label=f"Huber Derivative (δ={delta_param})",
    color="tab:red",
    lw=2,
    linestyle="--",
)
ax2.set_title("Loss Gradient ∂L/∂r vs. Residual r")
ax2.set_xlabel("Residual r = ŷ - y")
ax2.set_ylabel("Gradient Magnitude")
ax2.legend()

plt.tight_layout()
plt.show()

## 5. Outlier Sensitivity Experiment

We now train univariate models on two datasets:
1. `house_prices_toy.csv`: 8 clean samples.
2. `house_prices_outlier.csv`: 8 clean samples + 1 severe outlier (`area=6.0, price=25.0`).


In [ ]:
data_clean = np.loadtxt(Path("../data/house_prices_toy.csv"), delimiter=",", skiprows=1)
data_outlier = np.loadtxt(Path("../data/house_prices_outlier.csv"), delimiter=",", skiprows=1)


def train_linear_model(
    data: np.ndarray, loss_name: str, lr: float = 0.01, epochs: int = 400, delta: float = 1.5
) -> tuple[float, float]:
    w, b = 0.0, 0.0
    x = data[:, 0]
    y = data[:, 1]
    n = len(x)

    for _ in range(epochs):
        r = (w * x + b) - y
        if loss_name == "mse":
            dw = (2.0 / n) * np.sum(x * r)
            db = (2.0 / n) * np.sum(r)
        elif loss_name == "mae":
            s = np.sign(r)
            dw = (1.0 / n) * np.sum(x * s)
            db = (1.0 / n) * np.sum(s)
        elif loss_name == "huber":
            grad_r = np.where(np.abs(r) <= delta, r, delta * np.sign(r))
            dw = (1.0 / n) * np.sum(x * grad_r)
            db = (1.0 / n) * np.sum(grad_r)
        w -= lr * dw
        b -= lr * db
    return w, b


w_clean_mse, b_clean_mse = train_linear_model(data_clean, "mse")
w_corrupt_mse, b_corrupt_mse = train_linear_model(data_outlier, "mse")
w_corrupt_mae, b_corrupt_mae = train_linear_model(data_outlier, "mae", lr=0.03, epochs=600)
w_corrupt_huber, b_corrupt_huber = train_linear_model(
    data_outlier, "huber", lr=0.02, epochs=500, delta=1.0
)

print(f"Clean Data (MSE):           w={w_clean_mse:.3f}, b={b_clean_mse:.3f}")
print(f"Corrupted Data (MSE):       w={w_corrupt_mse:.3f}, b={b_corrupt_mse:.3f}")
print(f"Corrupted Data (MAE):       w={w_corrupt_mae:.3f}, b={b_corrupt_mae:.3f}")
print(f"Corrupted Data (Huber):     w={w_corrupt_huber:.3f}, b={b_corrupt_huber:.3f}")

In [ ]:
x_plot = np.linspace(2, 9, 100)

plt.figure(figsize=(9, 5))
plt.scatter(data_clean[:, 0], data_clean[:, 1], color="black", label="Nominal Data", s=40, zorder=4)
plt.scatter(
    [6.0], [25.0], color="red", s=90, marker="x", label="Injected Outlier (6.0, 25.0)", zorder=5
)

plt.plot(
    x_plot,
    w_clean_mse * x_plot + b_clean_mse,
    color="gray",
    linestyle=":",
    lw=2,
    label="Clean MSE Baseline",
)
plt.plot(
    x_plot,
    w_corrupt_mse * x_plot + b_corrupt_mse,
    color="tab:blue",
    lw=2,
    label="Corrupted MSE Fit",
)
plt.plot(
    x_plot,
    w_corrupt_mae * x_plot + b_corrupt_mae,
    color="tab:green",
    lw=2,
    label="Corrupted MAE Fit",
)
plt.plot(
    x_plot,
    w_corrupt_huber * x_plot + b_corrupt_huber,
    color="tab:red",
    lw=2,
    linestyle="--",
    label="Corrupted Huber Fit",
)

plt.title("Impact of Severe Outlier on Linear Regression Loss Objectives")
plt.xlabel("Area (x)")
plt.ylabel("Price (y)")
plt.ylim(0, 26)
plt.legend()
plt.tight_layout()
plt.show()

## 6. Summary

- **MSE** squares residual errors, imparting extreme sensitivity to corrupted or heavy-tailed distributions.
- **MAE** bounds gradient magnitude to $\pm 1$, making it robust against outliers but computationally stiff near zero residual.
- **Huber loss** combines the smoothness of MSE near the origin with the robust bounded gradients of MAE for large residuals.
